# Lecture 3 — How do we simulate a neutrino event?

### Neutrino Interactions, Simulation and Event Generation · *N. Kamp*

In Lectures 1 & 2 we built up the physics: **how neutrinos interact** (IBD → CCQE → DIS → Glashow)
and **what comes out** (electrons → EM showers, muons → MIPs + stochastic losses, taus → MIP + decay,
hadrons → hadronic showers, all seen via Cherenkov light).

Today we put it together as a **simulation pipeline** and generate real $\nu_\tau$ events:

```
   FLUX        →   INTERACTION   →   PROPAGATION   →   LIGHT      →   DETECTOR
 daemonflux       SIREN /            PROPOSAL          Cherenkov      photon hits,
 + nuSQuIDS       LeptonInjector     (lepton loss,     yield          event displays
 (atmo)           GENIE / NuGen      tau decay)        (Prometheus)
 power law        + weights
 (astro)
```

**How to use this notebook (Colab, ~1 hour):**
- Run cells top to bottom with `Shift+Enter`.
- Each stage works **even if a heavy install fails** — it falls back to a cached data file.
- Look for **🔧 Try it** boxes: change one number and re-run to see what happens.
- Nothing here needs a GPU.

## 0. Setup

This clones the helper repo and installs the *light* part of the stack (flux + injection + propagation).
The first run takes a few minutes. If a package fails to install, **keep going** — the cached fallbacks
cover every plot below.

In [ ]:
# --- Get the helper code (src/helpers.py, data/) ---
import os, sys, subprocess
REPO = 'https://github.com/USER/REPO.git'   # <-- point at your repo
if not os.path.exists('REPO'):
    # On Colab: clone. Locally: just use the current folder.
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1', REPO, 'REPO'], check=False)
        sys.path.insert(0, 'REPO/src')
    else:
        sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
        sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# --- Install the light stack (safe to fail; we fall back to cache) ---
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)

if 'google.colab' in sys.modules:
    pip('pyarrow', 'awkward')        # light, pure-wheel deps for reading cached data

    # --- SIREN (optional, for the live injection in §2) ---
    # The PyPI wheel lags the git repo, so the up-to-date build is from source.
    # Pick ONE; both are commented out by default so the lecture runs on cache.
    SIREN_WHEEL_URL = ''   # <-- BEST: prebuilt wheel you host (see README). pip(SIREN_WHEEL_URL)
    # if SIREN_WHEEL_URL: pip(SIREN_WHEEL_URL)
    # else: pip('cmake'); pip('git+https://github.com/Harvard-Neutrino/SIREN.git@<tag>')  # ~minutes

    # --- proposal / prometheus: NOT installed live (compile from source). ---
    # We display a cached Prometheus file in §5 instead.

    # NOTE: the flux stage needs NO package — it reads a small numpy table that
    # make_cache.py extracted from SIREN's Zenodo flux archive.

import numpy as np, matplotlib.pyplot as plt
import helpers as H
print('helpers loaded from:', H.__file__)
for m in ['siren', 'proposal', 'prometheus']:
    print(f'  {m:12s} available: {H.have(m)}')

> **About this Colab runtime.** You get Ubuntu + Python (check the printout above; Colab is on
> 3.11/3.12), with `numpy/scipy/matplotlib/pandas` pre-installed and an **ephemeral disk wiped each
> session** — so re-run setup every time. The physics tools are compiled C++, which is the slow part:
> - **`siren`** — we need the **git** version (newer than the PyPI wheel), so it **builds from
>   source** (needs `cmake` + a compiler; ~several min). For a smooth lecture, install a **prebuilt
>   wheel** instead (see the setup cell / README): build it once, host it, `pip install <url>`.
> - **`proposal`** — no PyPI wheel either, also compiles from source. **Prometheus** depends on it.
>
> Bottom line: we **don't build SIREN/Prometheus live in the lecture**. Every stage reads a small
> **cached** file (made once by `make_cache.py` in a full environment) and the notebook just analyses
> + displays it. The live-build code is shown but guarded. No GPU needed anywhere.

## 1. The flux — what's hitting the detector?

Two ingredients dominate a neutrino telescope's diet:

1. **Atmospheric** $\nu$ from cosmic-ray air showers — steep ($\sim E^{-3.7}$ at high energy), dominant below ~100 TeV.
2. **Astrophysical** $\nu$ — a near-isotropic diffuse flux, well described by a **single power law** $\propto E^{-\gamma}$.

We get the **atmospheric** flux straight from **SIREN's** packaged tables (Zenodo record 20129082) —
the same tables SIREN uses to weight injected events — and write the **astrophysical** one analytically.
No `daemonflux`/`nuflux`/`nuSQuIDS` install needed: `make_cache.py` already extracted a small numpy
table from SIREN's flux archive.

In [ ]:
# Atmospheric flux from SIREN's tables (surface + at-detector). The at-detector
# table already folds in nuSQuIDS oscillation + Earth absorption.
atmo = H.load_atmo_flux('atmo_flux_siren.npz')
Ea   = atmo['energy_gev']
cz   = atmo['coszen']
flav = list(atmo['flavors'])                # e.g. ['nue','numu','nutau']
imu  = flav.index('numu')
icz  = np.argmin(np.abs(cz - 0.0))          # ~horizontal
# flux_detector[coszen, energy, flavor, nu/nubar]; sum nu+nubar at the horizon
phi_atmo = atmo['flux_detector'][icz, :, imu, :].sum(axis=-1)

# --- Astrophysical: single power law (IceCube diffuse-like defaults) ---
phi_astro = H.astro_powerlaw(Ea, phi0=1.8e-18, gamma=2.5, e0=1e5)  # per flavor

plt.figure(figsize=(7,5))
plt.loglog(Ea, Ea**2*phi_astro, label=r'astro $\nu_\mu$  ($\gamma=2.5$)')
plt.loglog(Ea, Ea**2*phi_atmo,  label=r'atmospheric $\nu_\mu$ (horizon, at detector)')
plt.xlabel('Neutrino energy [GeV]'); plt.ylabel(r'$E^2\,d\Phi/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
plt.legend(); plt.title('The two fluxes a telescope sees'); plt.grid(alpha=.3); plt.show()

# Where do they cross? That crossover is why IceCube's astro signal lives above ~100 TeV.
cross = Ea[np.argmin(np.abs(phi_astro - phi_atmo))]
print(f'astro overtakes atmospheric near {cross/1e3:.0f} TeV')

> **🔧 Try it.** Change `gamma` from `2.5` to `2.0` (a harder spectrum) and re-run.
> Which direction does the atmospheric/astro crossover move, and why does a *harder* astro
> spectrum make high-energy events relatively more common?

### 1b. What the Earth does: surface vs at-detector

SIREN ships **both** the surface flux and the at-detector flux. The at-detector one was propagated
through the Earth with `nuSQuIDS` — so we *don't* run nuSQuIDS; we just plot the **ratio** of the two
tables and read off its two physical effects:

- **Oscillations** ($\sim$tens of GeV): the unique **$\nu_\tau$ appearance** that lets telescopes do
  atmospheric oscillation measurements (Lecture 1).
- **Earth absorption** (above $\sim$100 TeV): up-going high-energy $\nu$ get attenuated.

In [ ]:
# Detector/surface ratio for nu_tau across the (energy, zenith) plane.
itau = flav.index('nutau')
surf = atmo['flux_surface'][:, :, itau, :].sum(axis=-1)   # [coszen, energy]
det  = atmo['flux_detector'][:, :, itau, :].sum(axis=-1)
ratio = np.divide(det, surf, out=np.full_like(det, np.nan), where=surf > 0)

plt.figure(figsize=(7,5))
pc = plt.pcolormesh(Ea, cz, ratio, shading='auto', vmin=0, vmax=3, cmap='RdBu_r')
plt.xscale('log'); plt.xlabel('Energy [GeV]'); plt.ylabel(r'$\cos\theta_{zenith}$  (-1 = up-going)')
plt.colorbar(pc, label=r'$\nu_\tau$  detector / surface')
plt.title(r'Earth effects on atmospheric $\nu_\tau$: appearance (low E) + absorption (high E)')
plt.show()
print('ratio > 1 at low E = nu_tau appearing via oscillation; < 1 up-going at high E = absorption')

## 2. The interaction — inject neutrinos and let them scatter

Now the neutrinos hit the ice. Event generators handle the cross section + kinematics:

| Generator | Regime / role |
|---|---|
| **GENIE** (+ NuWro, GiBUU) | ~MeV–few GeV: QE, RES, nuclear effects |
| **NuGen** | IceCube's high-energy DIS workhorse |
| **LeptonInjector** | volume injection + weights, telescope-scale |
| **SIREN** | successor to LeptonInjector: flexible processes + weighting |

We use **SIREN** to inject high-energy $\nu_\tau$ CC events. The key telescope trick is
**weighted (forced) injection**: we *always* make an interaction in/near the detector and carry a
`generation weight`, instead of throwing away the ~$10^{12}$ neutrinos that would miss. (More in §6.)

In [ ]:
# ---- SIREN nu_tau CC-DIS injection (follows resources/examples/example1) ----
# We DON'T depend on running this live (SIREN compiles from source); the plots
# below read the cached injection make_cache.py produced with this same code.
if H.have('siren'):
    import siren
    from siren._util import GenerateEvents, SaveEvents

    events_to_inject = int(1e4)
    detector_model = siren.utilities.load_detector('IceCube')
    primary_type = siren.dataclasses.Particle.ParticleType.NuTau     # <- tau!
    Nucleon = siren.dataclasses.Particle.ParticleType.Nucleon

    # CC DIS cross sections (CSMS splines that ship with SIREN):
    primary_processes, _ = siren.utilities.load_processes(
        'CSMSDISSplines', primary_types=[primary_type], target_types=[Nucleon],
        isoscalar=True, process_types=['CC'])
    primary_cross_sections = primary_processes[primary_type]

    # Injector: forces an interaction in the detector volume + a generation weight.
    injector = siren.injection.Injector()
    injector.number_of_events = events_to_inject
    injector.detector_model = detector_model
    injector.primary_type = primary_type
    injector.primary_interactions = primary_cross_sections
    injector.primary_injection_distributions = [
        siren.distributions.PrimaryMass(0),
        siren.distributions.PowerLaw(2, 1e4, 1e7),            # E: 10 TeV - 10 PeV
        siren.distributions.IsotropicDirection(),
        siren.distributions.ColumnDepthPositionDistribution(
            600, 600.0, siren.distributions.LeptonDepthFunction())]
    events, gen_times = GenerateEvents(injector)

    # Weighter: turns each generated event into a physical rate for a chosen flux.
    weighter = siren.injection.Weighter()
    weighter.injectors = [injector]
    weighter.detector_model = detector_model
    weighter.primary_type = primary_type
    weighter.primary_interactions = primary_cross_sections
    weighter.primary_physical_distributions = [
        siren.distributions.PowerLaw(2, 1e4, 1e7),
        siren.distributions.IsotropicDirection()]
    print(f'generated {len(events)} nu_tau events; weighter ready')
    # SaveEvents(events, weighter, gen_times, output_filename='output/nutau')
else:
    print('siren not installed -- fine, we load the cached injection next.')

In [ ]:
# Load injected events for plotting: cached SIREN output, else synthetic stand-in.
import pandas as pd
try:
    inj = pd.read_parquet(H.cached_path('siren_nutau_injection.parquet'))
    print(f'loaded {len(inj)} cached SIREN events')
except FileNotFoundError:
    rng = np.random.default_rng(1)
    n = 20000
    inj = pd.DataFrame({'energy_gev': 10**rng.uniform(5, 7, n),
                        'bjorken_y': rng.beta(1.5, 4, n),   # DIS y, peaked low
                        'cos_zen': rng.uniform(-1, 1, n)})
    print('Using synthetic injection placeholder (swap in real SIREN output).')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(np.log10(inj['energy_gev']), bins=40); ax[0].set_xlabel(r'$\log_{10}(E_\nu/\mathrm{GeV})$')
ax[0].set_title('Injected energy spectrum (pre-weight)')
ax[1].hist(inj['bjorken_y'], bins=40); ax[1].set_xlabel('inelasticity y')
ax[1].set_title(r'Inelasticity: $E_{had}=y E_\nu$ goes to the cascade')
plt.show()

> **🔧 Try it.** The inelasticity $y$ sets how energy splits between the **hadronic cascade** ($yE_\nu$)
> and the **outgoing lepton** ($(1-y)E_\nu$). For a $\nu_\tau$, the lepton energy is what powers the
> *second* bang. Histogram $(1-y)\,E_\nu$ — that's the tau energy distribution feeding §3.

## 3. Propagation — the tau's race between flying and decaying

A $\nu_\tau$ CC event makes a $\tau$. Whether you see a **double bang** depends on the tau
**decay length** $L = \gamma c\tau \approx 50\,\mathrm{m} \times (E_\tau/\mathrm{PeV})$ being long enough to
separate the two cascades, but short enough to decay inside the detector. **PROPOSAL** handles the
charged-lepton propagation and stochastic losses (Lecture 2).

In [ ]:
E_tau = np.logspace(4, 7, 100)   # 10 TeV - 10 PeV
L = 50.0 * (E_tau / 1e6)         # mean decay length [m], lab frame

plt.figure(figsize=(7,5))
plt.loglog(E_tau, L)
plt.axhspan(10, 1000, alpha=.12, color='C2')   # roughly resolvable -> contained in IceCube
plt.axhline(125, ls='--', color='grey'); plt.text(1.2e4, 140, 'IceCube string spacing ~125 m')
plt.xlabel(r'$E_\tau$ [GeV]'); plt.ylabel('mean tau decay length [m]')
plt.title('Why double-bangs live around a PeV'); plt.grid(alpha=.3, which='both'); plt.show()

# With PROPOSAL you'd sample the *actual* decay point + stochastic losses:
if H.have('proposal'):
    print('proposal available — you can sample real tau decay points & continuous/stochastic losses here.')

> **🔧 Try it.** Below ~100 TeV the two bangs merge into one cascade (looks like $\nu_e$ CC);
> above ~10 PeV the tau often leaves the detector before decaying (looks like a track).
> What energy window gives the cleanest double-bang? Read it off the green band.

## 4–5. Light yield + detector response — run Prometheus, then *see* the event

**Prometheus** is the open-source telescope simulation: it takes the injected event, propagates the
leptons (PROPOSAL), generates Cherenkov light, and propagates photons to the modules. The full
IceCube chain uses **PPC** (GPU); the open-source CPU path uses a **parametric photon yield**
(`fennel`) — slower per photon but no GPU needed, perfect for Colab.

Because Prometheus (via PROPOSAL) **compiles from source**, we **don't build it live** — we load a
cached Prometheus output and display it. The config below is shown so you see how the pieces connect;
if no cached file exists we fall back to a clearly-labelled **synthetic** double-bang for development.

In [ ]:
# ---- How a Prometheus nu_tau run is configured (illustrative) ----
# Run this in make_cache.py (full environment), NOT live in Colab.
if H.have('prometheus'):
    from prometheus import Prometheus
    from prometheus.config import config
    config['run']['nevents'] = 50
    config['injection']['name'] = 'SIREN'              # or 'LeptonInjector'
    config['detector']['geo file'] = '<path>/icecube.geo'
    config['photon propagator']['name'] = 'olympus'    # CPU path; avoids GPU PPC
    # p = Prometheus(config); p.sim()   # writes a parquet of photon hits + truth
    print('Prometheus configured (CPU photon propagator). See make_cache.build_prometheus_events.')
else:
    print('prometheus not installed -- loading a cached/synthetic event below.')

In [ ]:
hits = geo = meta = None

# 1) Cached real Prometheus event (the preferred path)
try:
    hits, geo = H.load_prometheus_event('prometheus_nutau_example.parquet', event=0)
    meta = {'synthetic': False}
except FileNotFoundError:
    pass

# 2) Synthetic teaching placeholder
if hits is None:
    hits, geo, meta = H.synthetic_double_bang(e_nu_gev=3e6, inelasticity=0.25)
    print('** SYNTHETIC placeholder event ** (swap in a cached Prometheus file for the real thing)')

print('event is synthetic:', meta.get('synthetic'))

In [ ]:
ax = H.plot_event_display(hits, geo=geo,
                          title=r'$\nu_\tau$ CC double-bang  (time-coloured photon hits)')
plt.show()

### The full zoo of signatures (Lecture 2 callback)

Same machinery, different final states → different topologies. With real Prometheus output you can
render each of these from truth-labelled events:

| Signature | Origin | Energy reco quality |
|---|---|---|
| **Single cascade** | $\nu_e$ CC, all-NC | best (calorimetric, contained) |
| **Through-going track** | $\nu_\mu$ CC, muon exits | poor (only $dE/dx$ lower bound) |
| **Starting track** | $\nu_\mu$ CC inside | good (cascade + track) |
| **Double bang / double pulse** | $\nu_\tau$ CC | good if both bangs contained |
| **Dimuon** | charm / di-muon | specialised |

## 6. Event weights — from generated events to a physical rate

We injected events *uniformly-ish* in energy to populate all topologies. To predict what a detector
**actually sees**, each event carries a generation weight $w_{gen}$, and we reweight to a physical flux:

$$ \frac{dN}{dt} \;=\; \sum_{\text{events}} w_{gen}\;\times\;\Phi(E,\theta)\;\times\; T_{live} $$

**Why weighted generation wins:** one event sample → *any* flux model (change $\gamma$, swap astro for
atmospheric) by just changing $\Phi$ in the sum. No re-simulating the expensive photon propagation.

In SIREN this $\Phi$ lives in the **Weighter's** `primary_physical_distributions` (the §2 cell set it
to the injection spectrum). Closing the loop: we weight the **same events** to the real fluxes from §1
— an astro power law, and the SIREN atmospheric $\nu_\tau$ table — and see where one overtakes the other.

In [ ]:
# ---- (illustrative) the physical flux you'd hand SIREN's Weighter ----
if H.have('siren'):
    import siren
    astro_phys = siren.distributions.PowerLaw(2.5, 1e4, 1e7)   # astro: single power law
    # atmospheric: a tabulated flux built from the SAME table plotted in section 1
    #   atmo_phys = siren.distributions.TabulatedFluxDistribution(<E grid>, <flux>, ...)
    #   weighter.primary_physical_distributions = [atmo_phys, siren.distributions.IsotropicDirection()]
    print('Weighter physical flux: PowerLaw (astro) or TabulatedFluxDistribution (atmo, from section 1).')

In [ ]:
# ---- the actual reweighting, applied to the cached events ----
from scipy.interpolate import RegularGridInterpolator
E_ev  = inj['energy_gev'].to_numpy()
cz_ev = inj['cos_zen'].to_numpy()
w_gen = inj['gen_weight'].to_numpy() if 'gen_weight' in inj else np.full(len(inj), 1.0/len(inj))

def rate_curve(weights, label, **kw):
    h, edges = np.histogram(np.log10(E_ev), bins=30, weights=weights)
    c = 0.5*(edges[1:]+edges[:-1])
    plt.step(c, h, where='mid', label=label, **kw); return c, h

plt.figure(figsize=(7.5,5))
for gamma in (2.0, 2.5):                                  # astro, two spectral indices
    rate_curve(w_gen*H.astro_powerlaw(E_ev, gamma=gamma), f'astro $\\gamma={gamma}$')

# atmospheric nu_tau: interpolate the section-1 at-detector table at each event (E, cos_zen)
atmo = H.load_atmo_flux()
itau = list(atmo['flavors']).index('nutau')
det_tau = atmo['flux_detector'][:, :, itau, :].sum(-1)   # [coszen, energy]
interp = RegularGridInterpolator((atmo['coszen'], np.log10(atmo['energy_gev'])), det_tau,
                                 bounds_error=False, fill_value=0.0)
czc = np.clip(cz_ev, atmo['coszen'].min(), atmo['coszen'].max())
phi_atmo_ev = interp(np.column_stack([czc, np.log10(E_ev)]))
rate_curve(w_gen*phi_atmo_ev, r'atmospheric $\nu_\tau$ (at detector)', ls='--', color='k')

plt.yscale('log'); plt.xlabel(r'$\log_{10}(E_\nu/\mathrm{GeV})$')
plt.ylabel('expected rate [a.u.]'); plt.legend()
plt.title(r'Same events, real fluxes: astrophysical vs atmospheric $\nu_\tau$')
plt.show()
if atmo.get('synthetic'): print('(atmospheric flux is the SYNTHETIC placeholder -- swap in the SIREN table)')

> **🔧 Try it.** The dashed atmospheric $\nu_\tau$ curve crosses the astro curves somewhere — above
> that energy a $\nu_\tau$ is *more likely astrophysical than atmospheric*. Read off the crossover, then
> make the astro spectrum harder ($\gamma=2.0$): does the threshold for a clean tau-appearance search
> move? (Normalization is a.u. here — it's the *shape/crossover* that matters.)

## Wrap-up

You ran the whole pipeline: **flux → interaction → propagation → light → detector → weights**, and
produced a $\nu_\tau$ event display tying back to every signature from Lecture 2.

**Tools you touched / saw:** SIREN flux tables (nuSQuIDS oscillation baked in), SIREN/LeptonInjector
& GENIE (interaction), PROPOSAL (propagation), Prometheus + parametric Cherenkov (light),
weighted generation (rate).

**Go further:**
- Render a $\nu_e$ single cascade and a $\nu_\mu$ through-going track from the same Prometheus sample and compare.
- Put the §3 decay-length curve and the §5 display side by side: predict the topology before plotting it.
- Swap the astro flux for the oscillated atmospheric $\nu_\tau$ from §1b and recompute the rate.

**References:** Prometheus (arXiv:2304.14526), SIREN / LeptonInjector (arXiv:2012.10449),
PROPOSAL (Comput.Phys.Commun. 2019), nuSQuIDS (arXiv:2112.13804; engine behind the at-detector
tables), SIREN flux tables (Zenodo 20129082), Formaggio & Zeller (Rev.Mod.Phys. 2012).